# Wishart FGW + ConceptNet • Colab + Google Drive

Источник и результаты — [одна папка Google Drive](https://drive.google.com/drive/folders/1yE6U3uQvGULUl4g67zey3Hrc8a5NNXIo). Код основан на `feature/graphex-exact-v1` и расширен в `feature/graphex-exact-fgw-colab`.

**FGW сравнивает ego-подграфы с признаками, построенными из направленных гистограмм отношений ConceptNet, а не из текстовых эмбеддингов.** Вычисление FGW/POT — CPU; CUDA-ускоренный exact-кодек в другом ноутбуке не ускоряет FGW. Ограничение: не более 192 кандидатов на первом запуске; `max_nodes=10000` — пилот, исходный TSV остаётся полным. Вычисления идут на `/content`, контрольные точки — на Drive после каждого уровня.

In [ ]:
FOLDER_ID = "1yE6U3uQvGULUl4g67zey3Hrc8a5NNXIo"
BRANCH = "feature/graphex-exact-fgw-colab"
CONFIG = "configs/wishart_conceptnet_fgw_colab.yaml"
DATASET = "data/conceptnet_en_100k.tsv"
RUN_NAME = "wishart-fgw-cn100k-pilot-01"
MODE = "NEW"  # NEW or RESUME; use same RUN_NAME, config, input and git revision for RESUME
RUN_PIPELINE = True
CPU_WORKERS = 2

## 1. Среда Colab и проверка места

In [ ]:
import os, sys, shutil, subprocess
from pathlib import Path
assert sys.platform == "linux", "Ноутбук рассчитан на Colab Linux"
SCRATCH = Path("/content/semmap-fgw")
SCRATCH.mkdir(parents=True, exist_ok=True)
if shutil.disk_usage("/content").free < 3 * 1024**3:
    raise RuntimeError("Требуется минимум 3 GiB свободного места для staging")
os.environ["OMP_NUM_THREADS"] = "2"
os.environ["OPENBLAS_NUM_THREADS"] = "2"
os.environ["MKL_NUM_THREADS"] = "2"
print("Python", sys.version.split()[0], "CPU", os.cpu_count(), "Free GiB", round(shutil.disk_usage("/content").free / 1024**3,2))

## 2. Репозиторий и зависимости

На Python 3.13 не применяем устаревшие constraints, несовместимые с этим runtime. Для FGW обязателен POT.

In [ ]:
REPO = SCRATCH / "repo"
if not (REPO / ".git").exists():
    subprocess.run(["git","clone","--depth","1","--branch",BRANCH,"https://github.com/SemanticMap/semgraphex.git",str(REPO)],check=True)
else:
    subprocess.run(["git","-C",str(REPO),"fetch","origin",BRANCH],check=True)
    subprocess.run(["git","-C",str(REPO),"checkout","-B",BRANCH,"FETCH_HEAD"],check=True)
COMMIT = subprocess.check_output(["git","-C",str(REPO),"rev-parse","HEAD"],text=True).strip()
install = [sys.executable,"-m","pip","install","-q"]
if sys.version_info < (3,13):
    install += ["-c",str(REPO/"requirements/constraints.txt")]
install += ["-e",str(REPO)+"[wishart,notebook]","google-api-python-client","google-auth-httplib2"]
subprocess.run(install,check=True)
subprocess.run([sys.executable,"-c","import numpy, scipy, ot; print('numpy',numpy.__version__,'scipy',scipy.__version__,'POT',ot.__version__)"],check=True)
print("Repo commit:",COMMIT)

## 3. Находим папку по ID, а не по предположению о пути на Drive

In [ ]:
from google.colab import drive, auth
drive.mount("/content/drive")
auth.authenticate_user()
import google.auth
from googleapiclient.discovery import build
credentials,_ = google.auth.default(scopes=["https://www.googleapis.com/auth/drive.readonly"])
api = build("drive","v3",credentials=credentials,cache_discovery=False)
root_id = api.files().get(fileId="root",fields="id").execute()["id"]
parts=[]
current=FOLDER_ID
visited=set()
while current != root_id:
    if current in visited:
        raise RuntimeError("Цикл родителей в Drive")
    visited.add(current)
    item=api.files().get(fileId=current,fields="id,name,mimeType,parents",supportsAllDrives=True).execute()
    if item["mimeType"] != "application/vnd.google-apps.folder":
        raise RuntimeError("FOLDER_ID не папка")
    parts.append(item["name"])
    parents=item.get("parents",[])
    if len(parents)!=1:
        raise RuntimeError("Папка не находится в MyDrive: для shared drive требуется отдельный staging")
    current=parents[0]
DRIVE_ROOT=Path("/content/drive/MyDrive").joinpath(*reversed(parts))
if not DRIVE_ROOT.is_dir():
    raise FileNotFoundError(f"Папка по ID не видна в mounted MyDrive: {DRIVE_ROOT}")
SOURCE=DRIVE_ROOT/DATASET
if not SOURCE.is_file():
    raise FileNotFoundError(f"Исходный TSV отсутствует: {SOURCE}")
print("Verified Drive root:",DRIVE_ROOT)
print("Input:",SOURCE,"bytes:",SOURCE.stat().st_size)

## 4. Проверка FGW-конфигурации

FGW выполняется только на CPU; ограничение кандидатов должно соответствовать `transport_max_candidates`. Принудительно не переключаемся на Typed WL.

In [ ]:
import yaml
config_path=REPO/CONFIG
cfg=yaml.safe_load(config_path.read_text(encoding="utf-8"))
wish=cfg["wishart"]
assert wish["metric"] == "fgw"
assert wish["candidate_limit"] <= wish["transport_max_candidates"]
assert 0.0 <= wish["fgw_alpha"] <= 1.0
print({"metric":wish["metric"],"candidate_limit":wish["candidate_limit"],"landmark_rank":wish["transport_rank"],"alpha":wish["fgw_alpha"],"max_nodes":cfg["dataset"]["max_nodes"]})

## 5. Запуск или возобновление

NEW создаёт новый прогон; RESUME восстанавливается из последнего подтверждённого checkpoint. Не меняйте `RUN_NAME`, конфигурацию, исходный файл или коммит между прерыванием и RESUME. Результаты пишутся в `runs/RUN_NAME` указанной папки.

In [ ]:
if MODE not in {"NEW","RESUME"}:
    raise ValueError("MODE must be NEW or RESUME")
cmd=[sys.executable,"-m","semmap_haken.wishart_colab_cli",
     "--config",str(config_path),
     "--drive-root",str(DRIVE_ROOT),
     "--dataset-drive",DATASET,
     "--run-name",RUN_NAME,
     "--scratch-root",str(SCRATCH/"scratch"),
     "--device","cpu",
     "--cpu-workers",str(CPU_WORKERS)]
if MODE=="RESUME":
    cmd.append("--resume")
print("Running:", " ".join(cmd))
if RUN_PIPELINE:
    subprocess.run(cmd,cwd=str(REPO),check=True)
else:
    print("Dry run: RUN_PIPELINE=False")

## 6. Результаты и контрольные точки

In [ ]:
import json
run_dir=DRIVE_ROOT/"runs"/RUN_NAME
print("Run:",run_dir)
if run_dir.exists():
    print("Files:",[p.name for p in sorted(run_dir.iterdir())])
    for name in ["DRIVE_CHECKPOINT.json","COLAB_RUN.json","hierarchy.json"]:
        p=run_dir/name
        if p.is_file():
            print(name,p.read_text(encoding="utf-8")[:5000])
    print("COMPLETED:",(run_dir/"COMPLETED").is_file())
else:
    print("Результаты пока отсутствуют.")

## Что не меняется

Этот ноутбук выполняет повторную кластеризацию Wishart с FGW. Он **не перекодирует автоматически** уже существующие архивы Graphex Exact v1. Для применения exact-кодека к новым уровням после завершения FGW нужен отдельный прогон кодека на сохранённых `level_NNN` и `transition_NNN_NNN+1`.